In [ ]:
import os
from sentence_transformers import CrossEncoder
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import glob
from langchain_core.documents import Document
from typing import List, Dict, Any, Optional
import torch

from markdown_splitter import split_markdown_documents

In [ ]:
def load_markdown_files(pattern: str = "./docs/**/*.md") -> List[Document]:
    docs: List[Document] = []
    for path in glob.glob(pattern, recursive=True):
        with open(file=path, mode="r", encoding="utf-8") as f:
            text = f.read()
        docs.append(Document(page_content=text, metadata={"source": path}))
    return docs

In [ ]:
def get_embedding_model(
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
) -> HuggingFaceEmbeddings:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    )

In [ ]:
def build_chroma_from_documents(
    chunks: List[Document],
    persist_directory: str = "./chroma_db",
    collection_name: str = "markdown_collection",
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
) -> Chroma:
    embeddings = get_embedding_model(embedding_model_name)
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )
    return vector_store

In [ ]:
def load_chroma(
    persist_directory: str = "./chroma_db",
    collection_name: str = "markdown_collection",
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
) -> Chroma:
    embeddings = get_embedding_model(embedding_model_name)
    vector_store = Chroma(
        persist_directory=persist_directory,
        collection_name=collection_name,
        embedding_function=embeddings,
    )
    return vector_store

In [ ]:
def retrieve_candidates(
    vector_store: Chroma,
    query: str,
    top_k: int = 5,
    metadata_filter: Optional[Dict[str, Any]] = None,
) -> List[Document]:
    docs = vector_store.similarity_search(query, k=top_k, filter=metadata_filter)
    return docs

In [ ]:
def rerank_documents(
    query: str,
    docs: List[Document],
    reranker_name: str = "cross-encoder/ms-marco-MiniLM-L6-v2",
    top_n: int = 5,
) -> List[Dict[str, Any]]:
    if len(docs) == 0:
        return []

    reranker = CrossEncoder(reranker_name)
    pairs = [[query, d.page_content] for d in docs]
    scores = reranker.predict(pairs)
    ranked = []
    for d, s in zip(docs, scores):
        ranked.append({"document": d, "score": float(s)})

    ranked.sort(key=lambda x: x["score"], reverse=True)

    return ranked[:top_n]

In [ ]:
def build_context_from_ranked(
    ranked_docs: List[Dict[str, Any]], max_chars: int = 3000
) -> str:
    parts = []
    total = 0

    for i, item in enumerate(ranked_docs, start=1):
        d = item["document"]
        s = item["score"]
        source = d.metadata.get("source", "unknown")
        h1 = d.metadata.get("h1", "")
        h2 = d.metadata.get("h2", "")
        h3 = d.metadata.get("h3", "")

        block = (
            f"[{i}] score={s:.4f} source={source} h1={h1} | h2={h2} | h3={h3}\n"
            f"{d.page_content}\n"
        )

        if total + len(block) > max_chars:
            break

        parts.append(block)
        total += len(block)

    return "\n---\n".join(parts)

In [ ]:
def main():
    pattern = "./docs/**/*.md"
    persist_dir = "./chroma_db"
    collection = "markdown_collection"

    if not os.path.exists(persist_dir):
        print("Index not found. Building index...")
        raw_docs = load_markdown_files(pattern)
        print(f"Loaded files: {len(raw_docs)}")

        chunks = split_markdown_documents(raw_docs, chunk_size=400, chunk_overlap=60)
        print(f"Total chunks: {len(chunks)}")

        _ = build_chroma_from_documents(
            chunks=chunks,
            persist_directory=persist_dir,
            collection_name=collection,
            embedding_model_name=r"D:/modelscope/hub/models/microsoft/harrier-oss-v1-0___6b",
        )
        print("Chroma index built and persisted.")
    else:
        print("Using existing Chroma index.")

    vector_store = load_chroma(
        persist_directory=persist_dir,
        collection_name=collection,
        embedding_model_name="D:/modelscope/hub/models/microsoft/harrier-oss-v1-0___6b",
    )

    query = "Do you know how to use Python for text processing?"

    candidates = retrieve_candidates(vector_store=vector_store, query=query, top_k=5)

    print(f"Retrieved candidates: {len(candidates)}")

    ranked = rerank_documents(
        query=query,
        docs=candidates,
        reranker_name="D:/modelscope/hub/models/BAAI/bge-reranker-v2-m3",
        top_n=5,
    )

    print("\nTop reranked results:")
    for i, item in enumerate(ranked, start=1):
        d = item["document"]
        s = item["score"]
        print(f"[{i}] score={s:.4f}")
        print(f"source={d.metadata.get('source', 'unknown')}")
        print(
            f"h1={d.metadata.get('h1', '')} | h2={d.metadata.get('h2', '')} | h3={d.metadata.get('h3', '')}"
        )

    context = build_context_from_ranked(ranked_docs=ranked, max_chars=3000)
    print("\nFinal context preview:\n")
    print(context[:1200])


main()